Resultaten wegschrijven in genormaliseerd datamodel. Voorlopig gesimuleerd als sqlite. Connectie met LSVI databank of andere masterdata nog uit te klaren?

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sqlite3
from datetime import datetime
import pandas as pd

from arcgis.gis import GIS

import os
import sys

# Get the absolute path of the folder above the notebook
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from src import utils

### Connectie naar databank

In [3]:
# Aanmaken 
def init_database(db_path="lsvi_resultaten_slank.sqlite"):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("PRAGMA foreign_keys = ON;")
    
    # Tabel 1: WaarnemingEvent
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS waarneming_event (
        collectie_id TEXT PRIMARY KEY,
        global_id TEXT,
        user_name TEXT,
        bwk_plot_id TEXT,
        bwk_globalid TEXT,
        bwk_centroid_x REAL,
        bwk_centroid_y REAL,
        locatie_x REAL,
        locatie_y REAL,
        EPSG INTEGER,
        doel_habitattype TEXT,
        created_date DATETIME,
        last_edited_date DATETIME,
        timestamp_measurement DATETIME
    );
    """)
    
    # Tabel 2: Resultaat (Nu ook voor de losse soorten!)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS resultaat (
        resultaat_id INTEGER PRIMARY KEY AUTOINCREMENT,
        collectie_id TEXT,
        voorwaarde_id INTEGER,
        vraag_id TEXT,
        subvraag TEXT,
        waarde_tekst TEXT,
        waarde_numeriek REAL,
        FOREIGN KEY (collectie_id) REFERENCES waarneming_event(collectie_id) ON DELETE CASCADE
    );
    """)
    conn.commit()
    return conn

# Helperfunctie om ArcGIS Epoch Milliseconds om te zetten naar een ISO string
def format_agol_date(epoch_ms):
    if epoch_ms is None:
        return None
    # AGOL timestamps zijn in milliseconden, Python verwacht seconden
    return datetime.fromtimestamp(epoch_ms / 1000.0).strftime('%Y-%m-%d %H:%M:%S')



### Run ETL

In [4]:
# Uitvoering
# Joost to do: credentials uit key vault of environment
# Load secrets into environment
LOCAL_DB = "G:\\Mijn Drive\\keepass_db.kdbx"
ENTRY_TITLE = "AGOL"
AGOL_URL = "https://gisservices.inbo.be/portal"
FEATURE_LAYER_NAME = "xlsform_hab1-4"

AGOL_USER, AGOL_PASS = utils.load_keepass_credentials(
        db_path=LOCAL_DB, entry_name=ENTRY_TITLE
    )

today = datetime.now().strftime("%Y%m%d")
db_path = f"../output/lsvi_resultaten_{today}.sqlite"

# Verbinden met AGOL
print(f"Verbinden met {AGOL_URL}...")
gis = GIS(AGOL_URL, AGOL_USER, AGOL_PASS)

conn = init_database(db_path)
cursor = conn.cursor()

feature_layer_list = ['xlsform_hab1-4', 'xlsform_hab5-7', 'xlsform_hab9']

for FEATURE_LAYER_NAME in feature_layer_list:
    # Get feature layer id from name
    layer_items = gis.content.search(FEATURE_LAYER_NAME, item_type='Feature Layer', max_items=10)
    # Exclude item in list if ends with _form
    layer_items = [item for item in layer_items if not item.title.endswith('_form')]
    feature_layer_item_id = layer_items[0].id if layer_items else None
    print(f"Feature layer {feature_layer_item_id} downloaden...")
    feature_layer = layer_items[0].layers[0]

    # Vraag alle records op (1=1) inclusief WGS84 geometrie (out_sr=4326)
    features_result = feature_layer.query(where="1=1", out_sr=4326)
    print(f"Succesvol {len(features_result.features)} features gedownload. Start verwerking...")

    field_aliases = {
        f.name: f.alias for f in feature_layer.properties.fields
    } if hasattr(feature_layer, 'properties') else {}

    # Loop over each feature in feature layer
    for feature in features_result.features:
        geom = feature.geometry if feature.geometry else {}
        attrs = feature.attributes if feature.attributes else {}
        
        # Controleer of de cruciale collectie_id aanwezig is
        collectie_id = attrs.get('collectie_id')
        if not collectie_id:
            collectie_id = attrs.get('globalid')
            
        # Metadata extraheren en parsen
        global_id = attrs.get('globalid')
        user_name = attrs.get('username')
        habitat_keuze = attrs.get('habitat_keuze')
        bwk_plot_id = attrs.get('bwk_plot_id')
        bwk_globalid = attrs.get('bwk_globalid')
        bwk_centroid_x = attrs.get('bwk_centroid_x')
        bwk_centroid_y = attrs.get('bwk_centroid_y')
        
        created_date_txt = format_agol_date(attrs.get('datum'))
        last_edited_date_txt = format_agol_date(attrs.get('last_edited_date'))
        
        # Tijdstip van het bezoek bepalen (Datum-veld + Uur-veld combineren)
        datum_txt = format_agol_date(attrs.get('datum'))
        uur_txt = attrs.get('uur')  # string zoals "14:59"
        tijdstip_waarneming = f"{datum_txt.split(' ')[0]} {uur_txt}" if datum_txt and uur_txt else datum_txt

        # Geometrie
        x = geom.get('x')
        y = geom.get('y')
        epsg = geom.get('spatialReference', {}).get('wkid')

        # WaarnemingEvent wegschrijven
        cursor.execute("""
            INSERT INTO waarneming_event (
                collectie_id, 
                global_id, 
                user_name, 
                bwk_plot_id,
                bwk_globalid,
                bwk_centroid_x,
                bwk_centroid_y,
                locatie_x, 
                locatie_y, 
                epsg, 
                doel_habitattype,  
                created_date, 
                last_edited_date,
                timestamp_measurement
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(collectie_id) DO UPDATE SET
                global_id=excluded.global_id,
                user_name=excluded.user_name,
                bwk_plot_id=excluded.bwk_plot_id,
                bwk_globalid=excluded.bwk_globalid,
                bwk_centroid_x=excluded.bwk_centroid_x,
                bwk_centroid_y=excluded.bwk_centroid_y,
                locatie_x=excluded.locatie_x,
                locatie_y=excluded.locatie_y,
                epsg=excluded.epsg,
                doel_habitattype=excluded.doel_habitattype,
                created_date=excluded.created_date,
                last_edited_date=excluded.last_edited_date,
                timestamp_measurement=excluded.timestamp_measurement;
            """, (collectie_id, global_id, user_name, bwk_plot_id, bwk_globalid, bwk_centroid_x, bwk_centroid_y, x, y, epsg, habitat_keuze, created_date_txt, last_edited_date_txt, tijdstip_waarneming))
        
        # Resultaten wegschrijven
        # Eerst collectieID wissen voor overschrijven
        cursor.execute("DELETE FROM resultaat WHERE collectie_id = ?", (collectie_id,))

        # Loop dynamisch door alle kolommen van de feature
        for key, value in attrs.items():
            # Sla lege cellen, systeemenmerken en layout-hulpmiddelen over
            if value is None:
                continue

            # Extraheer het getal (VoorwaardeID) als de kolom begint met 'vrg_'
            voorwaarde_id = None
            habitattype = None
            subvraag = None

            if key.startswith("vrg_"):
                clean_key = key[4:]

                # Controleer of het een matrixvraag betreft
                if "_matrix_" in clean_key:
                    # Bv. '736_1310_zv_matrix_0' -> base_part = '736_1310_zv'
                    base_part = clean_key.split("_matrix_")[0]
                    # Ophalen van de weergavenaam (subvraag) uit onze lookup dict
                    subvraag = field_aliases.get(key)
                else:
                    base_part = clean_key
                    subvraag = None  # Geen matrix, dus geen subvraag

                # Ontleed voorwaarde_id en habitattype uit base_part (bv. '712_1310_zk')
                parts = base_part.split("_", 1)
                if parts[0].isdigit():
                    voorwaarde_id = int(parts[0])
                    if len(parts) > 1:
                        habitattype = parts[1]  # Bv. '1310_zk' of '1310_zv'
            
                # Gegevenstype bepalen voor waarde_numeriek
                waarde_numeriek = None
                if isinstance(value, (int, float)):
                    waarde_numeriek = float(value)
                elif isinstance(value, str):
                    try:
                        waarde_numeriek = float(value)
                    except ValueError:
                        pass

                # Afhandeling van select_multiple (komma-gescheiden waarden)
                if isinstance(value, str) and "," in value:
                    soorten = [s.strip() for s in value.split(",")]
                    for soort in soorten:
                        if soort:
                            cursor.execute("""
                                INSERT INTO resultaat (
                                    collectie_id, voorwaarde_id, vraag_id, subvraag, waarde_tekst, waarde_numeriek
                                )
                                VALUES (?, ?, ?, ?, ?, ?);
                            """, (collectie_id, voorwaarde_id, key, subvraag,  soort, waarde_numeriek))
                else:
                    # Normale vraag / enkele waarde
                    cursor.execute("""
                        INSERT INTO resultaat (
                            collectie_id, voorwaarde_id, vraag_id, subvraag, waarde_tekst, waarde_numeriek
                        )
                        VALUES (?, ?, ?, ?, ?, ?);
                    """, (collectie_id, voorwaarde_id, key, subvraag, str(value), waarde_numeriek))
    
print("ETL afgerond. De SQLite database is volledig up-to-date.")
conn.commit()
conn.close()
    

✅ Success: Credentials for 'AGOL' loaded into environment variables!
Verbinden met https://gisservices.inbo.be/portal...
Feature layer 55a62509e1c749118a9217622dd5fed4 downloaden...
Succesvol 4 features gedownload. Start verwerking...
Feature layer 6a7bd396d14b4e9088c64cef3119dcc5 downloaden...
Succesvol 1 features gedownload. Start verwerking...
Feature layer 83b650fed2ef4bd5b56006098b9987f2 downloaden...
Succesvol 4 features gedownload. Start verwerking...
ETL afgerond. De SQLite database is volledig up-to-date.


In [5]:
sqlite_path = "../output/lsvi_resultaten_20260727.sqlite"

conn = sqlite3.connect(sqlite_path)
results = pd.read_sql_query("SELECT * FROM waarneming_event LIMIT 10;", conn)
display(results)
conn.close()

,collectie_id,global_id,user_name,bwk_plot_id,bwk_globalid,bwk_centroid_x,bwk_centroid_y,locatie_x,locatie_y,EPSG,doel_habitattype,created_date,last_edited_date,timestamp_measurement
0,{6E9EE85B-D1F2-42BE-8B2B-09172A0F685B},{6E9EE85B-D1F2-42BE-8B2B-09172A0F685B},None,NaN,NaN,NaN,NaN,0.000000,0.000000,4326,1310_pol,2026-07-22 12:00:00,2026-07-22 10:11:02,2026-07-22 10:08
1,{94B23600-A89A-47BE-9A82-4FE9637BC3C3},{94B23600-A89A-47BE-9A82-4FE9637BC3C3},None,NaN,{42CD62CB-10B2-48B7-A8BB-0A2B5EE58859},5.518337e+12,6.627371e+14,0.000000,0.000000,4326,3130_aom,2026-07-22 12:00:00,2026-07-22 10:18:24,2026-07-22 10:15
2,{957EF476-BE5F-42E5-A505-49416CC55429},{957EF476-BE5F-42E5-A505-49416CC55429},None,,{8706A9FF-4A60-497E-87F5-EF853B21B8BC},4.340605e+13,6.651825e+14,3.898538,51.171811,4326,2330_dw,2026-07-24 12:00:00,2026-07-24 15:30:42,2026-07-24 15:06
3,{9D6829F0-B79C-4209-A428-65B93E547F6D},{9D6829F0-B79C-4209-A428-65B93E547F6D},None,NaN,NaN,NaN,NaN,0.000000,0.000000,4326,1310_pol,2026-07-22 12:00:00,2026-07-22 10:09:13,2026-07-22 10:08
4,{B041D71E-3E03-4E29-984E-BCE8BEECEC99},{B041D71E-3E03-4E29-984E-BCE8BEECEC99},None,NaN,{1C028B58-05CA-461E-BE3F-B82C78699F45},4.373163e+13,6.649814e+14,3.925937,51.163303,4326,6510_hu,2026-07-23 12:00:00,2026-07-23 16:22:09,2026-07-23 14:49
5,{0A04C1AB-977A-4A1B-86DE-BFDFA3B43F1D},{0A04C1AB-977A-4A1B-86DE-BFDFA3B43F1D},None,NaN,{69FC6534-0685-4376-95B2-CA660DE4CD40},4.157158e+13,6.619372e+12,3.780354,51.074271,4326,9160,2026-07-22 12:00:00,2026-07-22 16:23:03,2026-07-22 14:51
6,{55553AA7-DA85-425C-9D79-E71D21162336},{55553AA7-DA85-425C-9D79-E71D21162336},None,lsvi2016_JR0227,{DE37BBB9-1FCF-464D-903F-CA8F03E46D05},5.519687e+13,6.627374e+14,0.000000,0.000000,4326,9120,2026-07-22 12:00:00,2026-07-22 10:32:24,2026-07-22 10:29
7,{AF62A5D3-E57C-4CF8-8A6E-705A9DA3E1E0},{AF62A5D3-E57C-4CF8-8A6E-705A9DA3E1E0},None,NaN,{6050FF2D-E68A-4FAE-8160-D1004B0460D3},4.135351e+13,6.617712e+14,3.715779,50.979298,4326,91e0_sf,2026-07-23 12:00:00,2026-07-23 13:12:39,2026-07-23 10:22
8,{F10A34E2-1416-49A1-906E-FFD70A75808E},{F10A34E2-1416-49A1-906E-FFD70A75808E},None,NaN,{6050FF2D-E68A-4FAE-8160-D1004B0460D3},4.135351e+13,6.617712e+14,3.715779,50.979298,4326,91e0_sf,2026-07-23 12:00:00,2026-07-24 15:31:25,2026-07-23 13:22


In [7]:
conn = sqlite3.connect(sqlite_path)
results = pd.read_sql_query("SELECT * FROM resultaat WHERE collectie_id == '{AF62A5D3-E57C-4CF8-8A6E-705A9DA3E1E0}'", conn)
conn.close()
results

,resultaat_id,collectie_id,voorwaarde_id,vraag_id,subvraag,waarde_tekst,waarde_numeriek
0,276,{AF62A5D3-E57C-4CF8-8A6E-705A9DA3E1E0},100,vrg_100_91e0_sf,NaN,10_20perc,NaN
1,277,{AF62A5D3-E57C-4CF8-8A6E-705A9DA3E1E0},1385,vrg_1385_91e0_sf,NaN,10_20perc,NaN
2,278,{AF62A5D3-E57C-4CF8-8A6E-705A9DA3E1E0},548,vrg_548_91e0_sf_matrix_10,Zwart tandzaad,5_10perc,NaN
3,279,{AF62A5D3-E57C-4CF8-8A6E-705A9DA3E1E0},255,vrg_255_91e0_sf_matrix_0,Groeiklasse 1,5_10perc,NaN
4,280,{AF62A5D3-E57C-4CF8-8A6E-705A9DA3E1E0},255,vrg_255_91e0_sf_matrix_4,Groeiklasse 5,20_30perc,NaN
5,281,{AF62A5D3-E57C-4CF8-8A6E-705A9DA3E1E0},255,vrg_255_91e0_sf_matrix_6,Groeiklasse 7,20_30perc,NaN
6,282,{AF62A5D3-E57C-4CF8-8A6E-705A9DA3E1E0},255,vrg_255_91e0_sf_matrix_5,Groeiklasse 6,30_40perc,NaN
7,283,{AF62A5D3-E57C-4CF8-8A6E-705A9DA3E1E0},205,vrg_205_91e0_sf,NaN,1,1.0
8,284,{AF62A5D3-E57C-4CF8-8A6E-705A9DA3E1E0},2283,vrg_2283_91e0_sf,NaN,60_70perc,NaN
9,285,{AF62A5D3-E57C-4CF8-8A6E-705A9DA3E1E0},548,vrg_548_91e0_sf_matrix_4,Gele lis,f,NaN
